# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_json = dataset.metadata.to_json()
title = getattr(dataset.metadata, 'name', 'Dataset Name Unknown')
description = getattr(dataset.metadata, 'description', 'No dataset description.')
print(f"{title}: {description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references use their `@id` fields for clarity and reproducibility.

In [ ]:
# List all available record sets and their fields using their `@id`s.
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

print("Record Sets found in dataset (using @id):")
for rs_obj in dataset.metadata.record_sets:
    rs_id = rs_obj['@id']
    rs_name = rs_obj.get('name', 'N/A')
    print(f"- Record Set: {rs_name} (@id={rs_id})")
    print("  Fields:")
    for field in rs_obj.get('fields', []):
        if isinstance(field, dict):
            field_id = field.get('@id', 'N/A')
            field_name = field.get('name', 'N/A')
            print(f"    - {field_name} (@id={field_id})")
        elif isinstance(field, str):
            print(f"    - (@id={field})")

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Define a list of record set "@id"s (add more if there are multiple record sets):
record_sets = record_set_ids

# Load data from each record set into a DataFrame
dataframes = {}
for rs_id in record_sets:
    # Each record is already a dict with fields by @id
    records_list = list(dataset.records(record_set=rs_id))
    if records_list:
        df = pd.DataFrame(records_list)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for {rs_id}")
    else:
        print(f"No records found for {rs_id}")

# For demonstration, show columns of the first record set
first_rs_id = record_sets[0]
print(f"Columns present in record set {first_rs_id}:")
print(dataframes[first_rs_id].columns.tolist())
dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply example data processing: filter based on a numeric field, normalize it, and group by a categorical field.

➡️ **Choose field `@id`s visible in Section 2 or 3. Replace the variable as appropriate for your field, e.g., `'cr:peptideCount'`, `'cr:abundance_normalized'`, etc.**

In [ ]:
# Example: Assume the record set has fields '@id' cr:peptideCount (numeric) and cr:accession (categorical)
numeric_field = 'cr:peptideCount'  # CHANGE as needed based on actual field @id
group_field = 'cr:accession'       # CHANGE as needed based on actual field @id
record_set_id = first_rs_id        # use the main record set

df = dataframes[record_set_id]

# Convert numeric field to float (if possible, handle errors)
if numeric_field in df.columns:
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
else:
    print(f"Field {numeric_field} not found in DataFrame columns:", df.columns.tolist())

# Filter records: e.g., peptideCount > 10
threshold = 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalize numeric field (z-score)
norm_col = f"{numeric_field}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records (using z-score):")
display(filtered_df[[numeric_field, norm_col]].head())

# Group by group_field, if present
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean', 'count']).reset_index()
    print(f"Grouped data by {group_field}:")
    display(grouped_df.head())
else:
    print(f"Group field {group_field} not in columns; skipping group operation.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If two numeric fields available, scatterplot
numeric_field2 = 'cr:abundance_normalized'  # Replace with actual @id of a second numeric field if available
if numeric_field2 in df.columns:
    plt.figure(figsize=(6,6))
    sns.scatterplot(x=df[numeric_field], y=df[numeric_field2], alpha=0.6)
    plt.xlabel(numeric_field)
    plt.ylabel(numeric_field2)
    plt.title(f"{numeric_field} vs {numeric_field2}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR\^2 Croissant-powered metadata enables transparent, reproducible access and processing with `mlcroissant`.
- The analyzed record set provides insights into protein abundance and features, such as peptide counts and potentially normalized abundances.
- Outlier filtering, normalization, and grouping allow for downstream machine learning or statistical workflows.

Explore additional relationships by extending field @id references or creating domain-specific filters as needed!